<a href="https://colab.research.google.com/github/arshiii08/E-Commerce-Sales-Analysis-Dashboard/blob/main/e_commerce_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# -------------------------------
# 1. Load the Datasets
# -------------------------------

# Load sales data (adjust encoding if needed)
sales_df = pd.read_csv('01-Jan-2018_to_20-Mar-2023.csv', encoding='latin-1')
# Load demographics data
survey_df = pd.read_csv('survey.csv', encoding='latin-1')

# Display a sample of each dataset
print("Sales Data Sample:")
print(sales_df.head())
print("\nSurvey Data Sample:")
print(survey_df.head())

Sales Data Sample:
  Order Date             Order ID  \
0   01/21/18  113-4202476-2985066   
1   01/21/18  113-0543938-3949016   
2   01/28/18  113-7250323-7133011   
3   03/04/18  113-1945249-5483441   
4   03/04/18  113-1945249-5483441   

                                               Title      Category  \
0  Scott 1000 Sheets Per Roll Toilet Paper, 27 Ro...  TOILET_PAPER   
1  VIVA Choose-A-Sheet* Paper Towels, White, Big ...   PAPER_TOWEL   
2       Godel, Escher, Bach: An Eternal Golden Braid     ABIS_BOOK   
3  Utopia Towels Soft Cotton Machine Washable, Ex...         TOWEL   
4                                                NaN           NaN   

    ASIN/ISBN  UNSPSC Code     Website        Release Date  Condition  \
0  B01NBYY28W   14111704.0  Amazon.com                 NaN        new   
1  B01LFFGW5K   14111703.0  Amazon.com                 NaN        new   
2   465026850   55101500.0  Amazon.com  1979-05-31 0:00:01  used good   
3  B01DLD4YY0   52120000.0  Amazon.com       

In [ ]:
# -------------------------------
# 2. Prepare Sales Data
# -------------------------------

# Ensure order_date is parsed as datetime
sales_df['Order Date'] = pd.to_datetime(sales_df['Order Date'], errors='coerce')

# Aggregate sales metrics per customer:
# - Total Spending: Sum of Item Total per customer
# - Average Order Value: Mean of Item Total per customer
# - Order Count: Count of orders per customer
# - Last Order Date: The most recent order date per customer (for recency)
customer_sales = sales_df.groupby('Order ID').agg({
    'Item Total': ['sum', 'mean'],
    'Order Date': 'max',
    'Order ID': 'count'  # Assuming each order has a unique order_id
}).reset_index()

# Flatten the MultiIndex columns
customer_sales.columns = ['Order ID', 'total_spending', 'avg_order_value', 'last_order_date', 'order_count']

# Calculate recency: days since the last order relative to the latest order date in the dataset
reference_date = sales_df['Order Date'].max()
customer_sales['recency'] = (reference_date - customer_sales['last_order_date']).dt.days

print("\nAggregated Sales Data:")
print(customer_sales.head())


TypeError: agg function failed [how->mean,dtype->object]

In [ ]:
print(sales_df.columns)


Index(['Order Date', 'Order ID', 'Title', 'Category', 'ASIN/ISBN',
       'UNSPSC Code', 'Website', 'Release Date', 'Condition', 'Seller',
       'Seller Credentials', 'List Price Per Unit', 'Purchase Price Per Unit',
       'Quantity', 'Payment Instrument Type', 'Purchase Order Number',
       'PO Line Number', 'Ordering Customer Email', 'Shipment Date',
       'Shipping Address Name', 'Shipping Address Street 1',
       'Shipping Address Street 2', 'Shipping Address City',
       'Shipping Address State', 'Shipping Address Zip', 'Order Status',
       'Carrier Name & Tracking Number', 'Item Subtotal', 'Item Subtotal Tax',
       'Item Total', 'Tax Exemption Applied', 'Tax Exemption Type',
       'Exemption Opt-Out', 'Buyer Name', 'Currency', 'Group Name'],
      dtype='object')
